# Phase 4: Build Memory

## Step 13: Short-Term Conversation Memory

### Learning

- Working memory
- Conversation context
- Sliding windows
- Context budgets
- Conversation summarization
- Lossy compression
- Message prioritization
- Memory versus context

---

## Key Takeaways

- A context window isn't one thing — it's several competing pieces (system
  instructions, summary, recent messages, retrieved evidence, tool results,
  the current message) that all have to fit in one budget. Counting them
  separately is what makes the trade-off visible.
- When the budget runs out, **the application decides what to drop**, in a
  fixed priority order — not the model, and not "whatever happens to be
  oldest" by accident.
- Summarizing is lossy on purpose. The goal isn't to preserve everything, it's
  to preserve what still matters — which is why some facts (a decision, a
  deadline) get stored as structured state instead of trusted to prose.
- Updating a summary incrementally (existing summary + new messages) is
  cheaper and more stable than re-summarizing the whole conversation from
  scratch every time.
- Short-term memory is really **context management within one conversation**,
  not memory in a deeper sense — it still all disappears when the session
  ends. That gap is exactly what Step 14 (Long-Term Memory) exists to close.

---

## To do (mirrors the Roadmap 1:1)

1. Define an artificial context budget
2. Count every context component
3. Define a context priority order
4. Implement a sliding message window
5. Summarize older messages
6. Update the existing summary
7. Preserve critical information explicitly
8. Test information retention
9. Test summary corruption
10. Add a conversation-memory debugger

No retrieval infrastructure needed here (no Elasticsearch/Chroma) — this step
is about managing conversation context, not searching documents. Kept simple:
plain dicts and functions, a scripted demo conversation, no framework.

## 0. Environment Setup

Just the OpenAI client and a token counter — nothing else needed.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import config
importlib.reload(config)

import tiktoken
from openai import OpenAI
from pydantic import BaseModel

from config import OPENAI_API_KEY, MODEL_NAME

client = OpenAI(api_key=OPENAI_API_KEY)
encoding = tiktoken.get_encoding("cl100k_base")


def count_tokens(text):
    return len(encoding.encode(text or ""))

## 1. Define an Artificial Context Budget

> Choose a context budget smaller than the model's actual maximum -- this
> makes it easier to trigger and observe context-management behavior.

The roadmap's own example (8,000 / 1,000 / 7,000) is realistic for
production, but a real conversation would need hundreds of long messages to
exceed it -- too long for a notebook demo. We use a much smaller budget here
purely so a short example conversation can actually trigger archiving and
summarization.

In [2]:
MAX_CONTEXT_TOKENS = 300      # artificially small, see markdown above
RESERVED_OUTPUT_TOKENS = 100
AVAILABLE_INPUT_TOKENS = MAX_CONTEXT_TOKENS - RESERVED_OUTPUT_TOKENS

print("Max context:      ", MAX_CONTEXT_TOKENS, "tokens")
print("Reserved output:  ", RESERVED_OUTPUT_TOKENS, "tokens")
print("Available input:  ", AVAILABLE_INPUT_TOKENS, "tokens")

Max context:       300 tokens
Reserved output:   100 tokens
Available input:   200 tokens


## 2. Count Every Context Component

> Before each model request, count tokens for: system instructions,
> conversation summary, recent messages, retrieved documents, long-term
> memories, tool results, current user message. Display the allocation.

Steps 7-12 already cover *how* retrieved documents and (later, Step 14)
long-term memories get produced -- here they're just placeholder strings, so
this section can focus on counting and budgeting, not re-running retrieval.

In [3]:
def count_context_components(system_instructions, summary, recent_messages, retrieved_documents, long_term_memories, tool_results, current_message):
    components = {
        "system_instructions": system_instructions,
        "summary": summary,
        "recent_messages": "\n".join(m["content"] for m in recent_messages),
        "retrieved_documents": "\n".join(retrieved_documents),
        "long_term_memories": "\n".join(long_term_memories),
        "tool_results": "\n".join(tool_results),
        "current_message": current_message,
    }

    token_counts = {name: count_tokens(text) for name, text in components.items()}
    total = sum(token_counts.values())

    for name, tokens in token_counts.items():
        print(f"{name:<20} {tokens:>5} tokens")
    print(f"{'TOTAL':<20} {total:>5} tokens   (budget: {AVAILABLE_INPUT_TOKENS})")

    return token_counts

In [4]:
_ = count_context_components(
    system_instructions="You are a helpful assistant for ByteMage employees.",
    summary="",
    recent_messages=[{"content": "Hi, my name is Marcus."}],
    retrieved_documents=["ByteMage employees may take up to five sick days per month."],
    long_term_memories=[],  # nothing yet -- Step 14 fills this in
    tool_results=[],
    current_message="What's our sick leave policy?",
)

system_instructions     10 tokens
summary                  0 tokens
recent_messages          7 tokens
retrieved_documents     13 tokens
long_term_memories       0 tokens
tool_results             0 tokens
current_message          7 tokens
TOTAL                   37 tokens   (budget: 200)


## 3. Define a Context Priority Order

> 1. System and safety instructions, 2. Current user message, 3. Required
> tool results, 4. Retrieved evidence, 5. Recent conversation messages,
> 6. Long-term memory, 7. Older conversation summary. The application, not
> the model, should enforce this ordering.

`fit_to_budget` walks the components in this exact order, keeping each one
only if it still fits. Whatever comes last in priority is the first to get
dropped when the budget runs out -- here, that's the old summary.

In [5]:
CONTEXT_PRIORITY = [
    "system_instructions",
    "current_message",
    "tool_results",
    "retrieved_documents",
    "recent_messages",
    "long_term_memories",
    "summary",
]


def fit_to_budget(components, budget):
    """Keep components in priority order until the budget runs out.
    Returns which components were kept and which were dropped."""
    kept = {}
    dropped = []
    remaining = budget

    for name in CONTEXT_PRIORITY:
        text = components[name]
        tokens = count_tokens(text)
        if tokens <= remaining:
            kept[name] = text
            remaining -= tokens
        else:
            dropped.append(name)

    return kept, dropped

In [6]:
components = {
    "system_instructions": "You are a helpful assistant for ByteMage employees.",
    "current_message": "What is our sick leave policy, and can you also remind me what we discussed earlier?",
    "tool_results": "",
    "retrieved_documents": "ByteMage employees may take up to five sick days per month without additional approval.",
    "recent_messages": "user: Hi, my name is Marcus.\nassistant: Nice to meet you, Marcus!",
    "long_term_memories": "",
    "summary": "Earlier in this conversation: Marcus discussed budget options and a March 15 deadline for a marketing campaign, eventually choosing the $5,000 standard package over the $10,000 premium package after ruling it out as too expensive, and asked that all future summaries be written as bullet points instead of paragraphs.",
}

kept, dropped = fit_to_budget(components, AVAILABLE_INPUT_TOKENS)
print("Kept:   ", list(kept.keys()))
print("Dropped:", dropped)

Kept:    ['system_instructions', 'current_message', 'tool_results', 'retrieved_documents', 'recent_messages', 'long_term_memories', 'summary']
Dropped: []


## 4. Implement a Sliding Message Window

> Keep only the most recent messages verbatim -- e.g. the last six
> user-assistant turns. Remove older turns from the immediate prompt when the
> budget is exceeded. Store them in the conversation database rather than
> deleting them.

The roadmap's own example keeps six turns; this notebook uses two, purely so
a short demo conversation (Section 8) actually triggers archiving. "Removed
from the prompt" here means moved into `archived_messages` -- nothing is
deleted.

In [7]:
KEEP_LAST_N_TURNS = 2  # small on purpose -- see markdown above


def apply_sliding_window(all_messages, keep_last_n_turns):
    """Split into (archived_messages, recent_messages). One turn = one user
    message + one assistant message."""
    keep_count = keep_last_n_turns * 2
    if len(all_messages) <= keep_count:
        return [], all_messages
    return all_messages[:-keep_count], all_messages[-keep_count:]

In [8]:
example_messages = [
    {"role": "user", "content": "Hi, my name is Marcus."},
    {"role": "assistant", "content": "Nice to meet you, Marcus!"},
    {"role": "user", "content": "We're planning a marketing campaign."},
    {"role": "assistant", "content": "Great, what's the budget?"},
    {"role": "user", "content": "Either $10,000 or $5,000."},
    {"role": "assistant", "content": "Do you have a preference?"},
]

archived, recent = apply_sliding_window(example_messages, KEEP_LAST_N_TURNS)
print("Archived:", len(archived), "messages")
print("Recent:  ", len(recent), "messages (kept verbatim)")

Archived: 2 messages
Recent:   4 messages (kept verbatim)


## 5. Summarize Older Messages

> When messages move outside the recent window: send them to a summarization
> prompt, ask for a structured summary, and store important facts, decisions,
> user preferences, unresolved questions, and constraints. Replace the older
> messages in the prompt with the summary.

Structured output (same pattern as Steps 8/10/11/12) instead of free-text --
a `decisions: list[str]` field is much easier to inspect later (Section 9)
than trying to parse a decision back out of a paragraph.

In [9]:
class ConversationSummary(BaseModel):
    important_facts: list[str]
    decisions: list[str]
    user_preferences: list[str]
    unresolved_questions: list[str]
    constraints: list[str]


def summarize_messages(messages):
    conversation_text = "\n".join(f"{m['role']}: {m['content']}" for m in messages)

    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "Summarize this conversation excerpt into the structured fields "
                    "provided. Use short, specific bullet-style facts, not prose."
                ),
            },
            {"role": "user", "content": conversation_text},
        ],
        response_format=ConversationSummary,
    )
    return response.choices[0].message.parsed


def format_summary(summary):
    """Plain-text version of a ConversationSummary, for the prompt or for display."""
    if summary is None:
        return ""
    lines = []
    if summary.important_facts:
        lines.append("Facts: " + "; ".join(summary.important_facts))
    if summary.decisions:
        lines.append("Decisions: " + "; ".join(summary.decisions))
    if summary.user_preferences:
        lines.append("Preferences: " + "; ".join(summary.user_preferences))
    if summary.unresolved_questions:
        lines.append("Unresolved: " + "; ".join(summary.unresolved_questions))
    if summary.constraints:
        lines.append("Constraints: " + "; ".join(summary.constraints))
    return "\n".join(lines)

In [10]:
summary = summarize_messages(archived)
print(format_summary(summary))

Facts: User's name is Marcus


## 6. Update the Existing Summary

> Do not summarize the complete conversation from scratch every time. Provide
> the existing summary and the newly archived messages, and ask the model to
> produce an updated summary.

In [11]:
def update_summary(existing_summary, newly_archived_messages):
    existing_text = format_summary(existing_summary) if existing_summary else "(none yet)"
    new_text = "\n".join(f"{m['role']}: {m['content']}" for m in newly_archived_messages)

    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You maintain a running structured summary of a conversation. You are "
                    "given the EXISTING summary and NEWLY archived messages. Produce an "
                    "UPDATED summary that merges the new information in. Do not discard "
                    "existing facts unless the new messages explicitly change or resolve "
                    "them -- if a decision is revised, keep only the latest version."
                ),
            },
            {
                "role": "user",
                "content": f"EXISTING SUMMARY:\n{existing_text}\n\nNEWLY ARCHIVED MESSAGES:\n{new_text}",
            },
        ],
        response_format=ConversationSummary,
    )
    return response.choices[0].message.parsed

## 7. Preserve Critical Information Explicitly

> Mark certain data as durable within the current conversation: user's goal,
> selected option, important dates, decisions, constraints, unresolved
> tasks. Keep these in structured state rather than relying entirely on
> prose summarization.

The point of this section is trust, not automation: rather than asking a
model to *re-extract* these facts every turn (another model call, another
chance to drop something), the application just records them directly, in
plain code, at the moment they happen. A fact stored this way survives no
matter what the summarizer does with everything else.

In [12]:
def blank_critical_info():
    return {
        "user_goal": None,
        "selected_option": None,
        "important_dates": [],
        "decisions": [],
        "constraints": [],
        "unresolved_tasks": [],
    }

## 8. Test Information Retention

> Create a long conversation containing a person's name, a selected budget, a
> rejected option, a deadline, and a formatting preference. Continue chatting
> until early messages are summarized. Ask questions about the earlier
> information.

Six scripted turns. `add_turn` applies the sliding window after each one and
incrementally updates the summary (Sections 4-6) whenever a turn falls out of
the recent window. Critical facts (Section 7) are recorded directly as they
come up, alongside the summary -- not instead of it.

In [13]:
conversation = {
    "all_messages": [],
    "archived_messages": [],
    "recent_messages": [],
    "summary": None,
    "critical_info": blank_critical_info(),
}


def add_turn(state, user_message, assistant_message):
    state["all_messages"].append({"role": "user", "content": user_message})
    state["all_messages"].append({"role": "assistant", "content": assistant_message})

    archived, recent = apply_sliding_window(state["all_messages"], KEEP_LAST_N_TURNS)
    newly_archived = archived[len(state["archived_messages"]):]

    if newly_archived:
        state["summary"] = update_summary(state["summary"], newly_archived)
        print(f"-- archived {len(newly_archived)} message(s), summary updated --")

    state["archived_messages"] = archived
    state["recent_messages"] = recent

In [14]:
add_turn(
    conversation,
    "Hi, my name is Marcus and I'm planning a new marketing campaign.",
    "Nice to meet you, Marcus! What's the budget you're working with?",
)
conversation["critical_info"]["user_goal"] = "Plan a marketing campaign"

add_turn(
    conversation,
    "We're considering two options: a $10,000 premium package or a $5,000 standard package.",
    "Got it. Do you have a preference, or would you like a comparison of both?",
)

add_turn(
    conversation,
    "The $10,000 option is too expensive for us, so let's rule that out. We'll go with the $5,000 standard package.",
    "Understood -- we'll proceed with the $5,000 standard package.",
)
conversation["critical_info"]["selected_option"] = "$5,000 standard package"
conversation["critical_info"]["decisions"].append("Rejected the $10,000 premium package as too expensive")
conversation["critical_info"]["decisions"].append("Selected the $5,000 standard package")

add_turn(
    conversation,
    "Also, we need the campaign ready by March 15th -- that's a hard deadline.",
    "Noted -- March 15th deadline.",
)
conversation["critical_info"]["important_dates"].append("March 15 campaign deadline")

add_turn(
    conversation,
    "One more thing: please give me all future summaries as bullet points, not paragraphs.",
    "Sure, I'll use bullet points from now on.",
)
conversation["critical_info"]["constraints"].append("Write summaries as bullet points, not paragraphs")

-- archived 2 message(s), summary updated --
-- archived 2 message(s), summary updated --
-- archived 2 message(s), summary updated --


In [15]:
# Turn 6: don't script the assistant reply -- build the prompt from memory
# (summary + critical info + recent messages) and generate it for real, then
# check whether the early information survived.
current_message = "Can you remind me what we've decided so far, and how you'll format your answers?"

system_prompt = (
    "You are a helpful assistant. Use the conversation summary and critical facts below "
    "to answer the user's question.\n\n"
    f"SUMMARY:\n{format_summary(conversation['summary'])}\n\n"
    f"CRITICAL FACTS:\n{conversation['critical_info']}"
)

messages = [{"role": "system", "content": system_prompt}]
messages += conversation["recent_messages"]
messages.append({"role": "user", "content": current_message})

response = client.chat.completions.create(model=MODEL_NAME, messages=messages)
print(response.choices[0].message.content)

- You are planning a marketing campaign.
- Two budget options were considered: $10,000 premium package and $5,000 standard package.
- You rejected the $10,000 premium package as too expensive.
- You decided to proceed with the $5,000 standard package.
- The campaign must be ready by the hard deadline of March 15th.
- All future summaries and responses will be formatted using bullet points, not paragraphs.


Check the answer above against what actually happened early in the
conversation: does it correctly say **$5,000** (not $10,000), mention the
**March 15** deadline, and promise **bullet points**? Those facts came from
messages that are no longer in `recent_messages` at all -- they only survive
because they're in `summary` and `critical_info`.

## 9. Test Summary Corruption

> Intentionally create similar names, revised decisions, and contradictions.
> Observe whether the summary preserves the latest decision, accidentally
> merges facts, drops negations, or retains outdated information.

This is an empirical check, not something the code enforces -- run it and
read `summary.decisions` yourself. A good summary keeps only "Wednesday at
2pm" and "Sara Chin"; a corrupted one keeps both names, or reverts to
Tuesday.

In [16]:
corruption_test_messages = [
    {"role": "user", "content": "I'd like to book a consultation with Sarah Chen next week."},
    {"role": "assistant", "content": "Sure, I can help schedule that with Sarah Chen."},
    {"role": "user", "content": "Actually, I meant Sara Chin, a different person -- not Sarah Chen."},
    {"role": "assistant", "content": "Got it, I'll correct that to Sara Chin instead of Sarah Chen."},
    {"role": "user", "content": "Let's go with Tuesday at 10am."},
    {"role": "assistant", "content": "Tuesday at 10am it is."},
    {"role": "user", "content": "Actually, Tuesday doesn't work anymore -- let's do Wednesday at 2pm instead."},
    {"role": "assistant", "content": "Understood, switching to Wednesday at 2pm."},
]

corruption_summary = summarize_messages(corruption_test_messages)

print("Facts:    ", corruption_summary.important_facts)
print("Decisions:", corruption_summary.decisions)

Facts:     ['User wants to book a consultation with Sara Chin, not Sarah Chen.', 'Initial requested time was Tuesday at 10am.', 'User changed the appointment to Wednesday at 2pm.']
Decisions: ['Consultation booked with Sara Chin.', 'Appointment scheduled for Wednesday at 2pm.']


## 10. Add a Conversation-Memory Debugger

> Display: current summary, messages kept verbatim, messages archived, token
> budget, information marked as critical.

In [17]:
def print_memory_debug(state):
    print("=== Conversation Memory Debug ===")
    print("Token budget (input):", AVAILABLE_INPUT_TOKENS)

    print("\nCurrent summary:")
    print(format_summary(state["summary"]) or "(none yet)")

    print(f"\nMessages kept verbatim ({len(state['recent_messages'])}):")
    for m in state["recent_messages"]:
        print(f"  [{m['role']}] {m['content'][:60]}")

    print(f"\nMessages archived ({len(state['archived_messages'])}):")
    for m in state["archived_messages"]:
        print(f"  [{m['role']}] {m['content'][:60]}")

    print("\nCritical info:")
    for key, value in state["critical_info"].items():
        print(f"  {key}: {value}")


print_memory_debug(conversation)

=== Conversation Memory Debug ===
Token budget (input): 200

Current summary:
Facts: User's name is Marcus.; User is planning a new marketing campaign.; There are two options for the campaign budget: a $10,000 premium package or a $5,000 standard package.; Marcus decided to rule out the $10,000 option as too expensive.; Marcus will proceed with the $5,000 standard package.
Decisions: Marcus decided to go with the $5,000 standard package.
Preferences: Marcus prefers the less expensive $5,000 standard package over the $10,000 premium package.

Messages kept verbatim (4):
  [user] Also, we need the campaign ready by March 15th -- that's a h
  [assistant] Noted -- March 15th deadline.
  [user] One more thing: please give me all future summaries as bulle
  [assistant] Sure, I'll use bullet points from now on.

Messages archived (6):
  [user] Hi, my name is Marcus and I'm planning a new marketing campa
  [assistant] Nice to meet you, Marcus! What's the budget you're working w
  [user] We're 

### Reflection

- **What gets lost during summarization?** Anything not asked for in the
  structured schema, and anything the summarizer judges unimportant --
  exact wording, tone, and minor details are gone by design.
- **Should the summary update after every message?** No -- only when a
  message actually falls out of the recent window (Section 6). Updating on
  every turn would mean paying for a summarization call before it's needed.
- **Can summary errors become permanent?** Yes, once the original messages
  are outside the window and only the summary remains, a wrong merge or
  dropped fact has nothing to correct it against -- which is exactly why
  Section 7 keeps critical facts outside the summary entirely.
- **Should tool results be summarized?** Rarely — a tool result is often a
  precise value (a price, a status); summarizing it risks turning an exact
  fact into an approximate one. Better to keep it verbatim or drop it once
  it's no longer needed.
- **Which messages should never be removed?** The current message and system
  instructions (Section 3's priority order puts them first) — everything
  else is a candidate for archiving under budget pressure.
- **Is this real memory, or context management?** Context management --
  Section 0 says this outright. It only lasts the conversation; Step 14 is
  what makes facts survive across sessions.
- **How should revised decisions be represented?** As the *latest* value
  under one key, not two conflicting entries — Section 9's "Wednesday, not
  Tuesday" case is exactly what to check for.
- **Should the model set its own context budget?** No — Section 1's budget is
  a fixed application setting; letting the model decide would make context
  management as unpredictable as the thing it's supposed to control.
- **Can a summary reference something no longer in the conversation?** Yes --
  if an earlier fact gets revised or retracted, the summary needs to reflect
  the retraction, not just append the update alongside the stale fact.
- **How can summary quality be evaluated?** The same way Step 12 evaluates
  answers: give the summary to a human or LLM judge alongside the original
  messages, and check whether key facts survived and contradictions were
  resolved correctly.